In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir("..")

In [3]:
from sklearn.metrics import f1_score
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset
from vllm import TokensPrompt
from vllm import LLM
from pathlib import Path
from argparse import ArgumentParser

import torch
import json
import numpy as np

from transformers import AutoTokenizer
from tqdm.auto import tqdm
from datasets import load_dataset

from cluster_intrep_repo.utils import initialize_tokenizer, tokenize_blocksworld_generation, THINK_TOKEN
from cluster_intrep_repo.stacks_utils import *
from tqdm.auto import tqdm, trange

INFO 03-06 12:35:58 __init__.py:190] Automatically detected platform cuda.


In [4]:
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

compute_dtype = torch.bfloat16
device = 'cuda:1'
model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"

tokenizer = initialize_tokenizer(model_id)


In [5]:
n_blocks = 6

In [6]:
parsed_datasets = {
    4: "blocksworld-4-blocks-actions.json",
    6: "blocksworld-6-blocks-big-actions.json"
}

In [7]:
dataset = load_dataset(
    f"dmitriihook/deepseek-r1-qwen-32b-planning-{blocksworld_type[n_blocks]}")["train"]


with open(parsed_datasets[n_blocks], "r") as f:
    labels_dataset = json.load(f)

# labels_dataset = load_dataset(f"dmitriihook/blocksworld-6-self-probing-parsed")["train"]


def load_dataset_from_file(domain_name, task_name):
    prompt_dir = Path(f"./cot-planning/results/{domain_name}/deepseek-32b/")
    with open(prompt_dir / f"{task_name}.json", 'r') as file:
        return json.load(file)


task_name = "plan_generation_po"
domain_name = f"blocksworld_{n_blocks}_blocks"
eval_results = load_dataset_from_file(domain_name, task_name)["instances"]

eval_results = {x["dataset_idx"]: x for x in eval_results}

labels_dataset = {
    x["index"]: x for x in labels_dataset
}


In [8]:
n_rows = row_ns[n_blocks]

take_prob = 0.5

labels_dict = defaultdict(dict)

for idx, row in enumerate(dataset.select(range(n_rows))):
    generation = row["generation"]

    steps = generation.split("\n\n")

    labels = labels_dataset[idx]

    for line_n, (label, step) in enumerate(zip(labels["steps"], steps)):
        if not label["label"]:
            continue
        
        try:
            label = json.loads(label["label"])
        except:
            continue
        if "actions" not in label:
            continue
        
        if not label["actions"]:
            continue

        labels_dict[idx][line_n] = {
            "actions": label["actions"],
            "step": step
        }
            



In [9]:
# from transformers import AutoModelForCausalLM

# model     = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=compute_dtype, attn_implementation="sdpa", 
#                                                 device_map="auto")


In [10]:
# total_layers = model.config.num_hidden_layers
total_layers = 64

In [11]:
llm = LLM(model=model_id, task="reward", tensor_parallel_size=8)

INFO 03-06 12:36:14 config.py:1401] Defaulting to use mp for distributed inference
WARNING 03-06 12:36:14 arg_utils.py:1145] The model has a long context length (131072). This may cause OOM errors during the initial memory profiling phase, or result in low performance due to small KV cache space. Consider setting --max-model-len to a smaller value.
INFO 03-06 12:36:14 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='deepseek-ai/DeepSeek-R1-Distill-Qwen-32B', speculative_config=None, tokenizer='deepseek-ai/DeepSeek-R1-Distill-Qwen-32B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=8, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(gu

Loading safetensors checkpoint shards:   0% Completed | 0/8 [00:00<?, ?it/s]


INFO 03-06 12:36:39 model_runner.py:1115] Loading model weights took 7.5269 GB
(VllmWorkerProcess pid=2414407) INFO 03-06 12:36:40 model_runner.py:1115] Loading model weights took 7.5269 GB
(VllmWorkerProcess pid=2414404) INFO 03-06 12:36:40 model_runner.py:1115] Loading model weights took 7.5269 GB
(VllmWorkerProcess pid=2414434) INFO 03-06 12:36:40 model_runner.py:1115] Loading model weights took 7.5269 GB
(VllmWorkerProcess pid=2414422) INFO 03-06 12:36:41 model_runner.py:1115] Loading model weights took 7.5269 GB
(VllmWorkerProcess pid=2414417) INFO 03-06 12:36:41 model_runner.py:1115] Loading model weights took 7.5269 GB
(VllmWorkerProcess pid=2414412) INFO 03-06 12:36:41 model_runner.py:1115] Loading model weights took 7.5269 GB
(VllmWorkerProcess pid=2414427) INFO 03-06 12:36:41 model_runner.py:1115] Loading model weights took 7.5269 GB


(VllmWorkerProcess pid=2414412) (VllmWorkerProcess pid=2414422) (VllmWorkerProcess pid=2414417) (VllmWorkerProcess pid=2414427) (VllmWorkerProcess pid=2414434) (VllmWorkerProcess pid=2414404) (VllmWorkerProcess pid=2414407) INFO 03-06 13:29:01 multiproc_worker_utils.py:253] Worker exiting
INFO 03-06 13:29:01 multiproc_worker_utils.py:253] Worker exiting
INFO 03-06 13:29:01 multiproc_worker_utils.py:253] Worker exiting
INFO 03-06 13:29:01 multiproc_worker_utils.py:253] Worker exiting
INFO 03-06 13:29:01 multiproc_worker_utils.py:253] Worker exiting
INFO 03-06 13:29:01 multiproc_worker_utils.py:253] Worker exiting
INFO 03-06 13:29:01 multiproc_worker_utils.py:253] Worker exiting


In [12]:
batch_size = 200

last_hidden_states = []

for i in tqdm(range(0, n_rows, batch_size)):
    batch = dataset.select(range(i, min(i + batch_size, n_rows)))
    tokens = [tokenize_blocksworld_generation(
        tokenizer, row)[0] for row in batch]

    tokens = [TokensPrompt(prompt_token_ids=t) for t in tokens]

    output = llm.encode(tokens)

    for x in output:
        hs = x.outputs.data
        last_hidden_states.append(hs.cpu().to(torch.float16).numpy())


  0%|          | 0/10 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 200/200 [00:31<00:00,  6.31it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


In [13]:
collected_hidden_states = {total_layers -1 : last_hidden_states}

In [14]:
take_prob = 0.5

In [15]:
def make_data_to_process(dataset, n_rows, eval_results, answer_type, n_blocks, labels_dict, take_prob, tokenizer):
    data_to_process = []
    for idx, row in enumerate(tqdm(dataset.select(range(n_rows)))):
        if eval_results[idx]["llm_correct"] and answer_type == "incorrect":
            continue
        if not eval_results[idx]["llm_correct"] and answer_type == "correct":
            continue
        generation = row["generation"]

        # need to remove the eos token
        pos_pre = len(tokenize_blocksworld_generation(tokenizer, row, "")[0][:-1])

        for line_n, line in enumerate(generation.split("\n\n")):
            if line_n not in labels_dict[idx]:
                continue

            line_tokens = tokenizer.tokenize("\n\n" + line)[1:]
            pos_post = pos_pre + len(line_tokens)

            data_to_process.append({
                "idx": idx,
                "line_n": line_n,
                "pos_pre": pos_pre,
                "pos_post": pos_post,
                "step": labels_dict[idx][line_n]["step"],
                "actions": labels_dict[idx][line_n]["actions"]
            })

            pos_pre = pos_post

    return data_to_process


In [16]:
# data_correct = make_data_to_process(dataset, n_rows, eval_results, "correct", n_blocks, labels_dict, take_prob, tokenizer)
# data_incorrect = make_data_to_process(dataset, n_rows, eval_results, "incorrect", n_blocks, labels_dict, take_prob, tokenizer)
data_all = make_data_to_process(dataset, n_rows, eval_results, "all", n_blocks, labels_dict, take_prob, tokenizer)

  0%|          | 0/2000 [00:00<?, ?it/s]

In [ ]:
def process_data(item, action: tuple[str]):
    pos_pre = item["pos_pre"]
    pos_post = item["pos_post"]

    actions = [tuple(x) for x in item["actions"]]
    label = 0

    for other in actions:
        if other[0] == action[0] and other[1] == action[1]:
            label = 1
            break

    return {
        "pos_pre": pos_pre,
        "pos_post": pos_post,
        "label": label,
        "idx":  item["idx"],
        "line_n": item["line_n"],
        "label": label
    }

In [95]:
def process_data(items, target_action, drop_neg_prob=0.9):
    new_items = []

    for item in items:

        actions = [tuple(x) for x in item["actions"]]
        label = 0

        for other in actions:
            if other[0] == target_action[0] and other[1] == target_action[1]:
                label = 1
                break

        if label == 0 and np.random.rand() < drop_neg_prob:
            continue

        new_items.append({
            "pos_pre": item["pos_pre"],
            "pos_post": item["pos_post"],
            "label": label,
            "idx":  item["idx"],
            "line_n": item["line_n"],
            "label": label
        })

    return new_items
    

In [96]:
n_prev_tokens = 50


class StepProbeDataset(Dataset):
    def __init__(self, items, n_layer, target_action: tuple[str], drop_neg_prob):
        self.items = process_data(items, target_action, drop_neg_prob)
        self.hidden_states = collected_hidden_states[n_layer]
        self.n_layer = n_layer
        self.target_action = target_action

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]

        if True:
            pos_pre = item["pos_pre"]
            pos_post = item["pos_post"]

            label = item["label"]

            return {
                "input": self.hidden_states[item["idx"]][pos_post - n_prev_tokens:pos_post + 1],
                "labels": label
            }

            
        elif probe_type == "gru":
            post_pos = item["post_pos"]
            hidden_states = self.hidden_states[item["idx"]
                                               ][post_pos-n_prev_tokens:post_pos + 1]

            above, below = item["above"], item["below"]
            return {
                "input": hidden_states,
                "labels": state_to_label((above, below, None), self.top_block, self.bottom_block)
            }

In [ ]:
class StepProbe(torch.nn.Module):
    def __init__(self, input_size, hidden_size, n_blocks):
        super().__init__()
        # self.fc = torch.nn.Linear(input_size, hidden_size)
        # self.fc2 = torch.nn.Linear(hidden_size, n_blocks * (n_blocks + 2) * 2)
        # self.fc2 = torch.nn.Linear(input_size, n_blocks * (n_blocks + 2) * 2)
        self.fc2 = torch.nn.Linear(input_size, 2)
        # self.dropout = torch.nn.Dropout(0.1)

    def forward(self, x):
        # x = self.fc(x)
        # x = torch.nn.functional.relu(x)
        # x = self.dropout(x)
        x = self.fc2(x)
        return x
        # return x.view(-1, n_blocks + 2, n_blocks * 2)


class GRUProbe(torch.nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.gru = torch.nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = torch.nn.Linear(hidden_size, 2)

    def forward(self, x, *args):
        x, _ = self.gru(x)
        x = self.fc(x[:, -1])
        return x
    
class MultiProbe(torch.nn.Module):
    def __init__(self, input_size, hidden_size, n_probes):
        super().__init__()
        self.probes = torch.nn.ModuleList([torch.nn.Linear(input_size, hidden_size) for _ in range(n_probes)])
        self.fc = torch.nn.Linear(hidden_size, 2)

    def forward(self, x):
        
        for i in range(len(self.probes)):
            z_ = self.probes[i](x[:, i])
            if i == 0:
                z = z_
            else:
                z = z + z_
        z = z / len(self.probes)

        z = self.fc(z)

        return z
    
class AHProbe(torch.nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.q = torch.nn.Parameter(torch.randn(hidden_size))
        self.v = torch.nn.Parameter(torch.randn(hidden_size))

        self.proj = torch.nn.Linear(input_size, hidden_size)

        self.fc = torch.nn.Linear(hidden_size, 2)

    def forward(self, x, mask):
        x = self.proj(x)
        # scores = torch.einsum("bsh,h->bs", x, self.q)

        scores = x @ self.q

        # print(scores.shape, mask.shape)
        scores = scores.masked_fill(mask, -1000)
        scores = torch.nn.functional.softmax(scores, dim=-1)
        z = torch.matmul(scores.unsqueeze(1), x).squeeze(1)
        z = self.fc(z)
        return z
    
class MLHAProbe(torch.nn.Module):
    def __init__(self, input_size, hidden_size, n_heads):
        super().__init__()
        self.head_dim = hidden_size // n_heads
        self.n_heads = n_heads

        self.q = torch.nn.Parameter(torch.ones(n_heads, self.head_dim))
        self.v = torch.nn.Parameter(torch.ones(n_heads, self.head_dim))

        self.proj = torch.nn.Linear(input_size, hidden_size)

        self.fc = torch.nn.Linear(hidden_size, 2)

        # self.mlp = torch.nn.Sequential(
        #     torch.nn.Linear(hidden_size, hidden_size),
        #     torch.nn.ReLU(),
        #     torch.nn.Linear(hidden_size, 2)
        # )

    def forward(self, x, mask):
        x = self.proj(x)
        x = x.view(x.shape[0], x.shape[1], self.n_heads, self.head_dim)
        scores = torch.einsum("bshd,hd->bsh", x, self.q)

        scores = scores / np.sqrt(self.head_dim)

        # scores = scores.masked_fill(mask.unsqueeze(-1), -1000)
        scores = torch.nn.functional.softmax(scores, dim=-2)
        z = torch.einsum("bshd,bsh->bhd", x, scores)
        z = z.view(z.shape[0], -1)
        z = self.fc(z)
        return z


In [120]:
training_data = data_all[:50000]

train_test_split = 0.9
n_train = int(len(training_data) * train_test_split)

train_items = training_data[:n_train]
test_items = training_data[n_train:]


def collate_fn(batch):
    inputs = [torch.tensor(x["input"]) for x in batch]    
    masks = [torch.ones(x.shape[0], dtype=torch.bool) for x in inputs]
    inputs = pad_sequence(inputs, batch_first=True,
                          padding_value=0, padding_side="left")
    masks = pad_sequence(masks, batch_first=True,
                         padding_value=True, padding_side="left")
    labels = np.stack([x["labels"] for x in batch])
    labels = torch.tensor(labels, dtype=torch.int64)
    return {
        "input": inputs.to(device),
        "labels": labels.to(device),
        "mask": masks.to(device)
    }


def train_probe(probe, train_dataset, test_dataset, patience=30):
    optimizer = Adam(probe.parameters(), lr=1e-4)
    criterion = CrossEntropyLoss()
    train_loader = DataLoader(
        train_dataset, batch_size=128, shuffle=True, collate_fn=collate_fn)
    test_loader = DataLoader(
        test_dataset, batch_size=128, shuffle=False, collate_fn=collate_fn)

    n_epochs = 500
    best_f1 = float('-inf')
    early_stop_counter = 0

    for epoch in range(n_epochs):
        probe.train()
        total_loss = 0
        n_samples = 0

        for batch in train_loader:
            optimizer.zero_grad()
            input = batch["input"].to(device).float()
            labels = batch["labels"].to(device)
            mask = batch["mask"].to(device)

            output = probe(input, mask)

            # print(output.shape, labels.shape, input.shape)

            loss = criterion(output, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(batch["input"])
            n_samples += len(batch["input"])

        avg_train_loss = total_loss / n_samples

        # Evaluation
        probe.eval()
        with torch.no_grad():
            # block_wise_hits = np.zeros((n_blocks * 2), dtype=np.int64)
            block_wise_hits = 0
            total = 0
            val_loss = 0
            all_preds = []
            all_labels = []

            for batch in test_loader:
                input = batch["input"].to(device).float()
                labels = batch["labels"].to(device)
                mask = batch["mask"].to(device)

                output = probe(input, mask)

                loss = criterion(output, labels)
                val_loss += loss.item() * len(batch["input"])

                preds = output.argmax(dim=1)  # Assuming classification task
                hits = (preds == labels)

                block_wise_hits += hits.sum(dim=0).cpu().numpy()
                total += len(labels)

                all_preds.append(preds.cpu().numpy())
                all_labels.append(labels.cpu().numpy())

            block_wise_hits = block_wise_hits / total

            all_preds = np.concatenate(all_preds)
            all_labels = np.concatenate(all_labels)

            # Compute F1 score block-wise
            # block_wise_f1 = np.zeros(n_blocks * 2)
            # for i in range(n_blocks * 2):
            #     block_wise_f1[i] = f1_score(all_labels[:, i], all_preds[:, i], average='macro')

            # avg_f1 = block_wise_f1.mean()
            avg_f1 = f1_score(all_labels, all_preds, average='macro')

            val_loss /= total

            print(
                f"Epoch {epoch}, Train Loss: {avg_train_loss:.4f}, Hits: {block_wise_hits.mean():.4f}, F1: {avg_f1:.4f}, Val Loss: {val_loss:.4f}")

            # Early Stopping Check
            if avg_f1 > best_f1:
                best_f1 = avg_f1
                early_stop_counter = 0
            else:
                early_stop_counter += 1

            if early_stop_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch}")
                break

    return block_wise_hits, best_f1

In [121]:
top_block = "B"
bottom_block = "D"

top_blocks = ["B", "C", "A"]
bottom_blocks = ["D", "E", "F"]

# top_blocks = ["B"]  
# bottom_blocks = ["D"]

n_layer = total_layers - 1


train_dataset = StepProbeDataset(
   train_items, n_layer, ("stack", "A"), 0.6)
test_dataset = StepProbeDataset(train_items, n_layer, ("stack", "A"), 0.6)

print(len(train_dataset))

n_dim = 5120
probe = GRUProbe(n_dim, 500).to(device)
# if probe_type == "linear":
#     # pass
#     probe = StepProbe(n_dim, 500, n_blocks).to(device)
# elif probe_type == "gru":
#     probe = GRUProbe(n_dim, 1000, n_blocks).to(device)


21472


In [122]:

block_wise_hits, best_f1 = train_probe(
    probe, train_dataset, test_dataset, patience=10)

print(best_f1)

Epoch 0, Train Loss: 0.5949, Hits: 0.7488, F1: 0.5403, Val Loss: 0.5364
Epoch 1, Train Loss: 0.5298, Hits: 0.7546, F1: 0.5339, Val Loss: 0.4946
Epoch 2, Train Loss: 0.4846, Hits: 0.7962, F1: 0.6648, Val Loss: 0.4534
Epoch 3, Train Loss: 0.4345, Hits: 0.8130, F1: 0.7047, Val Loss: 0.4118
Epoch 4, Train Loss: 0.3773, Hits: 0.8517, F1: 0.7827, Val Loss: 0.3555
Epoch 5, Train Loss: 0.3166, Hits: 0.8962, F1: 0.8689, Val Loss: 0.3262
Epoch 6, Train Loss: 0.2563, Hits: 0.9240, F1: 0.9044, Val Loss: 0.2607
Epoch 7, Train Loss: 0.1926, Hits: 0.9137, F1: 0.8984, Val Loss: 0.2579
Epoch 8, Train Loss: 0.1417, Hits: 0.9439, F1: 0.9325, Val Loss: 0.1910
Epoch 9, Train Loss: 0.0990, Hits: 0.9258, F1: 0.9126, Val Loss: 0.2015
Epoch 10, Train Loss: 0.0664, Hits: 0.9576, F1: 0.9478, Val Loss: 0.1317
Epoch 11, Train Loss: 0.0478, Hits: 0.9496, F1: 0.9392, Val Loss: 0.1409
Epoch 12, Train Loss: 0.0360, Hits: 0.9499, F1: 0.9397, Val Loss: 0.1296
Epoch 13, Train Loss: 0.0265, Hits: 0.9214, F1: 0.9080, Val L

KeyboardInterrupt: 

In [125]:
print(dataset[0]["generation"])

Okay, so I have this block-stacking problem to solve. Let me try to figure out the steps needed to get from the initial state to the goal state. I'll take it step by step because I'm still getting the hang of these kinds of problems.

First, let me understand the initial conditions. The blocks are A, B, C, D, E, F. The hand is empty. The clear blocks are A, C, and E. That means these blocks have nothing on top of them and are on the table or on another block that's clear.

Looking at the initial setup:

- Block B is on top of F.
- Block C is on top of D.
- Block D is on top of B.
- So, the stack from bottom to top is F, B, D, C.
- Blocks A and E are on the table, clear.
- Block F is also on the table.

So, the initial stacks are:
- F (on table) → B → D → C
- A (on table)
- E (on table)

The goal is to have:
- A on top of E
- B on top of D
- C on top of B
- E on top of C
- F on top of A

Wait, that seems a bit confusing. Let me parse the goal again:

Goal:
- A is on top of E
- B is on t

In [ ]:
Epoch 10, Train Loss: 0.6021, Hits: 0.6850, F1: 0.6010, Val Loss: 0.5951 #1
Epoch 10, Train Loss: 0.4826, Hits: 0.7522, F1: 0.6794, Val Loss: 0.4949 #10
Epoch 10, Train Loss: 0.3880, Hits: 0.7905, F1: 0.7747, Val Loss: 0.4606 #20
Epoch 10, Train Loss: 0.3531, Hits: 0.8138, F1: 0.7964, Val Loss: 0.4153 #40, 5120
Epoch 10, Train Loss: 0.4523, Hits: 0.7524, F1: 0.6693, Val Loss: 0.4818 #40 1000
